In [1]:
import pandas as pd
import os
from pathlib import Path
from textblob import TextBlob
import re

In [2]:
#Define paths
BASE_DIR = Path("..").resolve()
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

In [3]:
#Load SemEval data
semeval = pd.read_csv(INTERIM_DIR / "semeval_task2_tc_merged.csv")
semeval.head()

,article_id,technique,start_char,end_char,source_file,text_content
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...


In [4]:
#Fill in missing techniques with 'Unknown'
semeval['technique'] = semeval['technique'].fillna('Unknown')

In [5]:
#Convert comma-separated technique strings to list of techniques
semeval['technique_list'] = semeval['technique'].astype(str).str.split(',')
semeval = semeval.explode('technique_list')
semeval['technique_list'] = semeval['technique_list'].str.strip()
semeval = semeval[semeval['technique_list'] != ""]
semeval.head()

,article_id,technique,start_char,end_char,source_file,text_content,technique_list
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Repetition
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Repetition
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Loaded_Language
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Causal_Oversimplification
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Loaded_Language


In [6]:
#Only show relevant text being evaluated for each row (using span on text)
semeval['span_text'] = semeval.apply(lambda row: row['text_content'][row['start_char']:row['end_char']], axis=1)
semeval.head()

,article_id,technique,start_char,end_char,source_file,text_content,technique_list,span_text
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Repetition,Islamization
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Repetition,Islamization
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Loaded_Language,outrage
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Causal_Oversimplification,"In order to convert to Islam, one says the sha..."
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Loaded_Language,egregious


In [7]:
def get_features(df):
    #Add Sentiment Score (-1.0 to 1.0): Propaganda often uses highly positive (Flag-Waving) or highly negative (Name-Calling) sentiment.
    df['sentiment'] = df['span_text'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

    #Add Punctuation Density: Excessive use of exclamation points or quotes often correlates with "Exaggeration" or "Doubt."
    df['punct_count'] = df['span_text'].apply(lambda x: len(re.findall(r'[!?"]', str(x))))

    #Add Lexical Diversity (Unique words / Total words): "Repetition" and "Slogans" have low lexical diversity.
    def lex_div(text):
        words = str(text).lower().split()
        if len(words) == 0: return 0
        return len(set(words)) / len(words)
    df['lexical_diversity'] = df['span_text'].apply(lex_div)

    #In the future, add Part-of-Speech (POS) Tags: High counts of adjectives and adverbs often signal "Loaded Language."

    return df

semeval = get_features(semeval)
semeval.head()

,article_id,technique,start_char,end_char,source_file,text_content,technique_list,span_text,sentiment,punct_count,lexical_diversity
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Repetition,Islamization,0.000000,0,1.000000
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Repetition,Islamization,0.000000,0,1.000000
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Loaded_Language,outrage,0.000000,0,1.000000
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Causal_Oversimplification,"In order to convert to Islam, one says the sha...",-0.166667,0,0.785714
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Loaded_Language,egregious,0.000000,0,1.000000


In [8]:
#Save to processed data folder
output_path = BASE_DIR / "data" / "processed" / "semeval_cleaned.csv"
semeval.to_csv(output_path, index=False)
print("SemEval DF is cleaned")

SemEval DF is cleaned
